# TPR / FPR Table (no manual confusion-matrix reading)

Reproduces the **TPR & FPR at the operating threshold** table (`tab:TRP-FPR`)
straight from the saved per-run artifacts — no opening confusion-matrix PNGs.

**How it works.** Each run's metrics dir holds `test_metrics.json` (with
`recall`, `precision`, `accuracy` at the chosen threshold) and an
`err_paths_*.csv` listing every misclassified sample, i.e. `E = FP + FN`.
Those four numbers determine the full confusion matrix exactly:

- `accuracy = (N - E) / N`  →  `N = E / (1 - accuracy)`
- `E = FP + FN = P[ recall(1-precision)/precision + (1-recall) ]`  →  solve `P` (positives)
- `TP = recall·P`, `FN = P - TP`, `FP = TP(1-precision)/precision`, `TN = (N-E) - TP`
- `TPR = TP/P` (= recall),  `FPR = FP/(N-P)`

Runs are discovered from the training logs (`*_seed{7-11}_run{1-5}`), the same
convention used by the other analysis notebooks. Stds use `ddof=1` (sample std)
to match the rest of the paper's tables.

In [1]:
import re, json, glob, os
import numpy as np
import pandas as pd

LOGS_DIR = "../results/logs"
SEEDS = [7, 8, 9, 10, 11]
RUN_OF_SEED = {s: i + 1 for i, s in enumerate(SEEDS)}
_SAVED_RE = re.compile(r"Saved test metrics to (\S+test_metrics\.json)")

# Model selection mirrors the main performance table (tab:ModelMetrics):
#   GS7 = threshold-7.0 runs, GS6 = threshold-6.0 (*_thr6) runs.
GROUPS = {
    "CNN3D":     {"GS7": "cnn_v0",            "GS6": "cnn_v0_thr6"},
    "ResNet3D":  {"GS7": "resnet_v0",         "GS6": "resnet_v0_thr6"},
    "RadioDino": {"GS7": "radiodino_v5_flat", "GS6": "radiodino_v5_thr6"},
    "OmniRad":   {"GS7": "omnirad_v5_flat",   "GS6": "omnirad_v5_thr6"},
}

In [2]:
def metrics_json_from_log(log_name):
    """Path of the test_metrics.json this run wrote (last match wins)."""
    with open(os.path.join(LOGS_DIR, log_name + ".log"), encoding="utf-8", errors="replace") as fh:
        matches = _SAVED_RE.findall(fh.read())
    if not matches:
        raise ValueError(f"No 'Saved test metrics to' line in {log_name}.log")
    return matches[-1]


def recover_confusion(recall, precision, accuracy, E):
    """Exact confusion matrix from scalar metrics + error count E = FP + FN."""
    N = round(E / (1 - accuracy))
    P = round(E / (recall * (1 - precision) / precision + (1 - recall)))
    TP = round(recall * P)
    FN = P - TP
    FP = round(TP * (1 - precision) / precision)
    TN = (N - E) - TP
    return {"N": N, "P": P, "Neg": N - P,
            "TP": TP, "FP": FP, "FN": FN, "TN": TN,
            "TPR": TP / P, "FPR": FP / (N - P)}


def load_run(log_name):
    """Recover TPR/FPR (and confusion counts) for a single run."""
    mjson = metrics_json_from_log(log_name)
    mdir = os.path.dirname(mjson)
    blob = json.load(open(mjson))
    m = blob["metrics"]
    err_files = glob.glob(os.path.join(mdir, "err_paths_*.csv"))
    if len(err_files) != 1:
        raise RuntimeError(f"Expected 1 err_paths csv in {mdir}, found {len(err_files)}")
    with open(err_files[0]) as fh:
        E = sum(1 for _ in fh) - 1  # drop header row
    cm = recover_confusion(m["recall"], m["precision"], m["accuracy"], E)
    # sanity: recomputed scalars must match the saved ones
    assert abs(cm["TP"] / cm["P"] - m["recall"]) < 1e-3
    assert abs((cm["TP"] + cm["TN"]) / cm["N"] - m["accuracy"]) < 1e-3
    cm["seed"] = blob["seed"]
    cm["log"] = log_name
    return cm

## Main results

TPR/FPR at the operating threshold for the four models, seeds 7–11. GS7 uses the
threshold-7.0 runs (`cnn_v0`, `resnet_v0`, `omnirad_v5_flat`, `radiodino_v5_flat`)
and GS6 the threshold-6.0 `*_thr6` runs. Stds use `ddof=1`.


In [3]:
# Per-run recovery for every (model, cohort, seed).
rows = []
for model, cohorts in GROUPS.items():
    for cohort, group in cohorts.items():
        for s in SEEDS:
            cm = load_run(f"{group}_seed{s}_run{RUN_OF_SEED[s]}")
            cm.update(model=model, cohort=cohort)
            rows.append(cm)

per_run = pd.DataFrame(rows)
per_run[["model", "cohort", "seed", "TP", "FP", "FN", "TN", "TPR", "FPR"]]

,model,cohort,seed,TP,FP,FN,TN,TPR,FPR
0,CNN3D,GS7,7,70,58,5,91,0.933333,0.389262
1,CNN3D,GS7,8,51,34,24,115,0.680000,0.228188
2,CNN3D,GS7,9,59,44,16,105,0.786667,0.295302
3,CNN3D,GS7,10,53,17,22,132,0.706667,0.114094
4,CNN3D,GS7,11,52,24,23,125,0.693333,0.161074
5,CNN3D,GS6,7,89,35,25,75,0.780702,0.318182
6,CNN3D,GS6,8,89,34,25,76,0.780702,0.309091
7,CNN3D,GS6,9,88,56,25,55,0.778761,0.504505
8,CNN3D,GS6,10,92,27,22,83,0.807018,0.245455
9,CNN3D,GS6,11,82,25,32,85,0.719298,0.227273


In [4]:
# Aggregate to mean ± std (ddof=1) percentages, one row per model/cohort.
def pct(series):
    return 100 * series

agg = (per_run
       .assign(TPR=lambda d: pct(d.TPR), FPR=lambda d: pct(d.FPR))
       .groupby(["model", "cohort"])
       .agg(TPR_mean=("TPR", "mean"), TPR_std=("TPR", lambda x: x.std(ddof=1)),
            FPR_mean=("FPR", "mean"), FPR_std=("FPR", lambda x: x.std(ddof=1)))
       .round(0).astype(int))
agg

TPR_mean  TPR_std  FPR_mean  FPR_std
model     cohort                                      
CNN3D     GS6           77        3        32       11
          GS7           76       11        24       11
OmniRad   GS6           81        6        33        9
          GS7           66        6        16        9
RadioDino GS6           79        2        28        8
          GS7           70        5        22        3
ResNet3D  GS6           77        3        29        6
          GS7           78       10        27        8

In [5]:
# Emit the LaTeX body rows (TPR over FPR in each \makecell, GS7 | GS6).
MODEL_ORDER = ["CNN3D", "ResNet3D", "RadioDino", "OmniRad"]

def cell(model, cohort):
    r = agg.loc[(model, cohort)]
    return (f"\\makecell{{$%d \\pm %d$ \\\\ $%d \\pm %d$}}"
            % (r.TPR_mean, r.TPR_std, r.FPR_mean, r.FPR_std))

for i, model in enumerate(MODEL_ORDER):
    shade = "\\rowcolor{rowgray}\n" if i % 2 == 0 else ""
    print(f"{shade}{model}")
    print(f"  & {cell(model, 'GS7')}")
    print(f"  & {cell(model, 'GS6')} \\\\\n")

\rowcolor{rowgray}
CNN3D
  & \makecell{$76 \pm 11$ \\ $24 \pm 11$}
  & \makecell{$77 \pm 3$ \\ $32 \pm 11$} \\

ResNet3D
  & \makecell{$78 \pm 10$ \\ $27 \pm 8$}
  & \makecell{$77 \pm 3$ \\ $29 \pm 6$} \\

\rowcolor{rowgray}
RadioDino
  & \makecell{$70 \pm 5$ \\ $22 \pm 3$}
  & \makecell{$79 \pm 2$ \\ $28 \pm 8$} \\

OmniRad
  & \makecell{$66 \pm 6$ \\ $16 \pm 9$}
  & \makecell{$81 \pm 6$ \\ $33 \pm 9$} \\

